# U-net

In [1]:
import math
import os
import random
import time
from datetime import datetime

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

from typing import Sequence


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, tv_tensors
from torchvision.transforms import v2
import torchvision.transforms.v2.functional as TF

In [2]:
# Seed every RNG so runs are comparable: CIFAR-10 seed variance (~±0.3-0.7%)
# is the same magnitude as many single-change effects.
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

## Model

### U-net

In [3]:
class Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # padding=1 keeps the spatial size through each conv. The 1x1-crop U-Net of the
        # paper shrinks 128px inputs to nothing by depth 4 (128 -> 124 -> 62 -> 58 -> 29
        # -> 25 -> 12 -> 8 -> 4 -> bottleneck needs 4-4=0), so "same" convs it is.
        # BatchNorm was not in the 2015 paper but makes the deeper stacks trainable at lr=1e-3.
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


In [4]:
class Encoder(nn.Module):
    def __init__(self, in_channels, base_channels=64, depth=4):
        super().__init__()

        self.hidden_channels = tuple(base_channels * (2 ** i) for i in range(depth)) # (64, 128, 256, 512)

        channels = (in_channels, *self.hidden_channels) # (in_channels, 64, 128, 256, 512)

        self.blocks = nn.ModuleList([
            Block(in_channels=channels[i], out_channels=channels[i + 1],)
            for i in range(depth)
        ])

        self.down_scale = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip_features = [] # gather skipped features for decoder

        for block in self.blocks:
            x = block(x)
            # learnt features from conv block
            skip_features.append(x)
            x = self.down_scale(x)
        return x, skip_features

In [5]:
class Decoder(nn.Module):
    def __init__(self, bottleneck_channels=1024, base_channels=64, depth=4):
        super().__init__()

        skip_channels = tuple(reversed(tuple(base_channels * (2 ** i) for i in range(depth))))

        channels = (bottleneck_channels, *skip_channels,) # (1024, 512, 256, 128, 64)

        # in = channels[i + 1] * 2 because the upconv output is concatenated with the skip,
        # and both carry channels[i + 1] channels.
        self.blocks = nn.ModuleList([
            Block(in_channels=channels[i + 1] * 2, out_channels=channels[i + 1],)
            for i in range(depth)
        ])

        self.up_convs = nn.ModuleList(
            [
                nn.ConvTranspose2d(channels[i], channels[i + 1], kernel_size=2, stride=2)
                for i in range(depth)
            ]
        )

    def forward(self, x, skip_features):
        for upconv, block, skip in zip(
            self.up_convs,
            self.blocks,
            reversed(skip_features),
        ):

            x = upconv(x)
            encoder_feature = self.copy_and_crop(skip, x)
            x = torch.cat([x, encoder_feature], dim=1)
            x = block(x)

        return x

    def copy_and_crop(self, skip_feature, decoder_feature):
        # With "same" convs and an input divisible by 2 ** depth the sizes already agree;
        # the crop is the fallback for odd input sizes.
        target_size = decoder_feature.shape[-2:]

        if skip_feature.shape[-2:] == target_size:
            return skip_feature

        return TF.center_crop(skip_feature, list(target_size))


In [6]:
class Unet(nn.Module):
    def __init__(self, in_channels, num_classes, base_channel=64, depth=4):
        super().__init__()

        self.encoder = Encoder(in_channels, base_channel, depth)

        encoder_channels = tuple(base_channel * (2 ** i) for i in range(depth))
        bottleneck_channels = base_channel * (2 ** depth) # 1024

        self.bottle_neck = Block(in_channels=encoder_channels[-1], out_channels=bottleneck_channels)

        self.decoder = Decoder(bottleneck_channels, base_channel, depth)

        self.fc = nn.Conv2d(base_channel, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        # The encoder returns (downsampled features, skips) - both are needed.
        x, skip_features = self.encoder(x)
        x = self.bottle_neck(x)
        x = self.decoder(x, skip_features)
        return self.fc(x)


## Data Loading

In [7]:
# Hyper parameters
# Data
IMAGE_SIZE = 128
CHANNELS = 3
NUM_CLASSES = 1       # Binary segmentation: pet vs background -> 1 logit per pixel

# U-Net architecture
BASE_CHANNELS = 64
DEPTH = 4
DROP_RATE = 0.0

# Training
BATCH_SIZE = 32
EPOCHS = 100
WARMUP_EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

In [8]:
class PreparePetSample:
    def __call__(self, image, mask):
        image = tv_tensors.Image(image)
        mask = tv_tensors.Mask(mask)
        return image, mask

class ConvertToBinaryMask:
    def __call__(self, image, mask):
        # Oxford trimap -> Binary mask
        # Oxford trimap: 1 = pet, 2 = background, 3 = boundary
        # Binary mask: 0 = background, 1 = pet/boundary
        # tv_tensors.Mask is already (1, H, W), so no unsqueeze - adding one here is what
        # produced the (B, 1, 1, H, W) batches.
        mask = (mask != 2).float()

        return image, mask

train_transform = v2.Compose([

    PreparePetSample(),

    # Resize slightly larger before randomly cropping.
    v2.Resize((144, 144), antialias=True),

    # Applied identically to both image and mask.
    v2.RandomCrop((IMAGE_SIZE, IMAGE_SIZE)),

    v2.RandomHorizontalFlip(),

    # Converts image to float32 [0, 1].
    # Keeps the mask as an integer mask until binary conversion.
    v2.ToDtype({
        tv_tensors.Image: torch.float32,
        tv_tensors.Mask: torch.long,
    }, scale=True),

    ConvertToBinaryMask(),
])


test_transform = v2.Compose([

    PreparePetSample(),

    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),

    v2.ToDtype({
        tv_tensors.Image: torch.float32,
        tv_tensors.Mask: torch.long,
    }, scale=True),

    ConvertToBinaryMask(),
])

train_dataset = datasets.OxfordIIITPet(
    root="dataset/",
    split="trainval",
    target_types="segmentation",
    transforms=train_transform,
    download=True,
)

test_dataset = datasets.OxfordIIITPet(
    root="dataset/",
    split="test",
    target_types="segmentation",
    transforms=test_transform,
    download=True,
)

In [9]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    generator=torch.Generator().manual_seed(SEED),
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

eval_train_dataset = datasets.OxfordIIITPet(
    root="dataset/",
    split="trainval",
    target_types="segmentation",
    transforms=test_transform,
    download=True,
)

eval_train_loader = DataLoader(
    dataset=torch.utils.data.Subset(
        eval_train_dataset,
        range(1000),
    ),
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

In [10]:
images, masks = next(iter(train_loader))

print(images.shape)          # [B, 3, 128, 128]
print(masks.shape)           # [B, 1, 128, 128]
print(torch.unique(masks))   # tensor([0., 1.])

torch.Size([32, 3, 128, 128])
torch.Size([32, 1, 128, 128])
tensor([0., 1.])


## Training

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick the fastest kernels

set_seed()  # seed right before weight init so the run is reproducible

cuda NVIDIA GeForce RTX 5080


### Unet

In [12]:
unet = Unet(in_channels=CHANNELS, num_classes=NUM_CLASSES, base_channel=BASE_CHANNELS, depth=DEPTH).to(device)

unet = unet.to(memory_format=torch.channels_last)

In [13]:
unet

Unet(
  (encoder): Encoder(
    (blocks): ModuleList(
      (0): Block(
        (block): Sequential(
          (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (5): ReLU(inplace=True)
        )
      )
      (1): Block(
        (block): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(128, eps=1e-

In [14]:
# One logit per pixel -> binary cross entropy, not CrossEntropyLoss.
# (CrossEntropyLoss expects C >= 2 class logits and an integer target without the channel
# dim; with NUM_CLASSES = 1 it would compare a single channel against itself. Label
# smoothing is a classification trick and does not apply here.)
criterion = nn.BCEWithLogitsLoss()

# No weight decay on 1-D params (BatchNorm gains/biases and conv biases) - decaying them
# just fights the normalisation.
decay, no_decay = [], []
for name, p in unet.named_parameters():
    (no_decay if p.ndim == 1 else decay).append(p)

optimizer = optim.AdamW(
    [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

# 5-epoch linear warmup (1e-4 -> 1e-3) guards against early divergence,
# then cosine decay to 0. Stepped once per epoch in run().
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=0.0)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

### Metrics

In [15]:
# Segmentation metrics, not classification accuracy. Pixel accuracy alone is misleading
# (background dominates), so Dice and IoU are the numbers to read.
#   pixel_acc - fraction of correctly labelled pixels
#   dice      - 2|P n G| / (|P| + |G|), averaged per image
#   iou       - |P n G| / |P u G|, aggregated over the dataset
@torch.inference_mode()
def evaluate(loader, model, threshold=0.5, eps=1e-7):
    model.eval()

    num_correct = 0
    num_pixels = 0
    dice_sum = 0.0
    inter_sum = 0.0
    union_sum = 0.0
    num_images = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True).float()

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(x)

        # Single logit per pixel -> sigmoid + threshold, not argmax over channels.
        preds = (torch.sigmoid(logits.float()) > threshold).float()

        num_correct += (preds == y).sum().item()
        num_pixels += y.numel()

        dims = (1, 2, 3)
        intersection = (preds * y).sum(dims)
        pred_sum = preds.sum(dims)
        target_sum = y.sum(dims)

        dice_sum += ((2 * intersection + eps) / (pred_sum + target_sum + eps)).sum().item()
        inter_sum += intersection.sum().item()
        union_sum += (pred_sum + target_sum - intersection).sum().item()
        num_images += y.size(0)

    return {
        "pixel_acc": num_correct / num_pixels,
        "dice": dice_sum / num_images,
        "iou": inter_sum / max(union_sum, eps),
    }


# Scalars alone do not show whether the masks look right - log a few image/GT/prediction
# triplets so TensorBoard's IMAGES tab shows the masks sharpening over training.
@torch.inference_mode()
def log_predictions(writer, model, loader, epoch, max_images=8):
    model.eval()

    x, y = next(iter(loader))
    x = x[:max_images].to(device, non_blocking=True, memory_format=torch.channels_last)
    y = y[:max_images].to(device, non_blocking=True).float()

    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
        logits = model(x)

    preds = (torch.sigmoid(logits.float()) > 0.5).float()

    # Row 1 = input, row 2 = ground truth, row 3 = prediction (masks expanded to 3 channels).
    panel = torch.cat([
        x.float().cpu(),
        y.cpu().repeat(1, 3, 1, 1),
        preds.cpu().repeat(1, 3, 1, 1),
    ], dim=0)

    writer.add_images("Predictions/image_gt_pred", panel.clamp(0, 1), epoch)

### Run

In [16]:
def run(model, optimizer, scheduler, criterion, epochs, writer, run_name, log_images_every=5):
    # AMP: forward in reduced precision, gradients rescaled to avoid underflow.
    # enabled= flags keep the whole loop runnable on CPU too.
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    os.makedirs("checkpoints", exist_ok=True)
    best_dice = 0.0
    step = 0

    try:
        for epoch in range(epochs):
            t0 = time.perf_counter()
            running_loss = 0.0
            model.train()

            for data, targets in train_loader:
                data = data.to(device, non_blocking=True, memory_format=torch.channels_last)
                # BCEWithLogitsLoss wants float targets of the same shape as the logits.
                targets = targets.to(device, non_blocking=True).float()

                # Forward propagation
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    scores = model(data)
                    loss = criterion(scores, targets)

                # Backward propagation
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()

                if step % 50 == 0:
                    writer.add_scalar("Loss/train_batch", loss.item(), step)
                step += 1

            avg_epoch_loss = running_loss / len(train_loader)

            # Train metrics on the clean fixed subset, test on the full test set.
            train_metrics = evaluate(eval_train_loader, model)
            test_metrics = evaluate(test_loader, model)

            # Step the scheduler once per epoch
            scheduler.step()

            # Log the learning rate used for the next epoch
            current_lr = optimizer.param_groups[0]["lr"]

            epoch_time = time.perf_counter() - t0
            writer.add_scalar("Loss/train_epoch", avg_epoch_loss, epoch)
            for split, metrics in (("train", train_metrics), ("test", test_metrics)):
                writer.add_scalar(f"PixelAcc/{split}", metrics["pixel_acc"], epoch)
                writer.add_scalar(f"Dice/{split}", metrics["dice"], epoch)
                writer.add_scalar(f"IoU/{split}", metrics["iou"], epoch)
            writer.add_scalar("Dice/gap", train_metrics["dice"] - test_metrics["dice"], epoch)
            writer.add_scalar("LR", current_lr, epoch)
            writer.add_scalar("Time/epoch_sec", epoch_time, epoch)

            if epoch % log_images_every == 0 or epoch == epochs - 1:
                log_predictions(writer, model, test_loader, epoch)

            # Keep the best weights, selected on Dice - cosine-to-zero means the last epoch
            # is usually the best, but this is insurance against surprises.
            if test_metrics["dice"] > best_dice:
                best_dice = test_metrics["dice"]
                torch.save({
                    "epoch": epoch,
                    "test_dice": test_metrics["dice"],
                    "test_iou": test_metrics["iou"],
                    "test_pixel_acc": test_metrics["pixel_acc"],
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                }, f"checkpoints/{run_name}_best.pt")

            print(
                f"Epoch [{epoch + 1}/{epochs}], "
                f"Loss: {avg_epoch_loss:.4f}, "
                f"Train Dice: {train_metrics['dice']:.4f}, "
                f"Test Dice: {test_metrics['dice']:.4f}, "
                f"Test IoU: {test_metrics['iou']:.4f}, "
                f"Test PixAcc: {test_metrics['pixel_acc']:.4f}, "
                f"LR: {current_lr:.6f}, "
                f"{epoch_time:.1f}s"
            )
    finally:
        torch.save(model.state_dict(), f"checkpoints/{run_name}_final.pt")
        writer.close()

    print(f"Best test dice: {best_dice:.4f} (checkpoints/{run_name}_best.pt)")

#### Run - Unet

In [17]:
import json
from torch.utils.tensorboard import SummaryWriter

# One directory per run so TensorBoard curves never overlap between runs.
run_name = f"{datetime.now():%Y%m%d-%H%M%S}_Unet_simple"
writer = SummaryWriter(f"runs/oxford_pet/{run_name}")
writer.add_text(
    "config",
    json.dumps(
        {
            # Model
            "architecture": "U-Net (same-padding + BatchNorm)",
            "task": "binary segmentation (pet vs background)",
            "dataset": "Oxford-IIIT Pet (trainval 3680 / test 3669)",
            "image_size": IMAGE_SIZE,
            "channels": CHANNELS,
            "num_classes": NUM_CLASSES,
            "base_channels": BASE_CHANNELS,
            "depth": DEPTH,
            "drop_rate": DROP_RATE,
            "params_millions": round(sum(p.numel() for p in unet.parameters()) / 1e6, 2),

            # Training
            "optimizer": "AdamW (no decay on 1-D params)",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "scheduler": f"warmup{WARMUP_EPOCHS}+cosine",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "loss": "BCEWithLogitsLoss",
            "augmentation": "resize144+randomcrop128+hflip",
            "seed": SEED,
        },
        indent=2,
    ),
)

2026-08-21 17:04:18.400586: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-21 17:04:18.428690: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-21 17:04:19.021341: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-21 17:04:19.021532: I external/local_xla/xla/tsl/cuda/

In [18]:
run(unet, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
    epochs=EPOCHS, writer=writer, run_name=run_name)

Epoch [1/100], Loss: 0.4460, Train Dice: 0.8146, Test Dice: 0.8146, Test IoU: 0.7116, Test PixAcc: 0.8503, LR: 0.000280, 9.6s
Epoch [2/100], Loss: 0.3546, Train Dice: 0.8290, Test Dice: 0.8293, Test IoU: 0.7330, Test PixAcc: 0.8580, LR: 0.000460, 6.2s
Epoch [3/100], Loss: 0.3072, Train Dice: 0.8619, Test Dice: 0.8602, Test IoU: 0.7782, Test PixAcc: 0.8898, LR: 0.000640, 6.2s
Epoch [4/100], Loss: 0.2809, Train Dice: 0.8693, Test Dice: 0.8641, Test IoU: 0.7843, Test PixAcc: 0.8989, LR: 0.000820, 6.2s
Epoch [5/100], Loss: 0.2688, Train Dice: 0.8436, Test Dice: 0.8445, Test IoU: 0.7540, Test PixAcc: 0.8687, LR: 0.001000, 6.3s
Epoch [6/100], Loss: 0.2551, Train Dice: 0.8575, Test Dice: 0.8527, Test IoU: 0.7586, Test PixAcc: 0.8864, LR: 0.001000, 6.1s
Epoch [7/100], Loss: 0.2360, Train Dice: 0.8602, Test Dice: 0.8617, Test IoU: 0.7791, Test PixAcc: 0.8853, LR: 0.000999, 6.1s
Epoch [8/100], Loss: 0.2258, Train Dice: 0.8900, Test Dice: 0.8822, Test IoU: 0.8039, Test PixAcc: 0.9101, LR: 0.00099

#### Run - resnet

In [19]:
# import json
# from torch.utils.tensorboard import SummaryWriter

# # One directory per run so TensorBoard curves never overlap between runs.
# run_name = f"{datetime.now():%Y%m%d-%H%M%S}_resnet34"
# writer = SummaryWriter(f"runs/cifar10/{run_name}")
# writer.add_text("config", json.dumps({
#     "arch": "resnet34-cifar", "layers": [3, 4, 6, 3],
#     "optimizer": "AdamW", "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, 
#     "scheduler": "warmup5+cosine", "epochs": epochs, "batch_size": BATCH_SIZE,
#     "label_smoothing": 0.1, "augmentation": "crop-reflect+hflip+erasing", "seed": SEED,
# }))

# run(net, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
#     epochs=epochs, writer=writer, run_name=run_name)

## Plot Accuracy

In [20]:
# plt.plot(train_accuracies, label='Train Accuracy')
# plt.plot(test_accuracies, label='Test Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.title('Training and Test Accuracy')
# plt.show()

## Benchmark

### Goal

Oxford-IIIT Pet — binary segmentation (pet vs background), Dice / IoU

There is no widely-tracked leaderboard for Oxford-IIIT Pet *segmentation* (the dataset is
mostly used for fine-grained classification and as a teaching set for U-Net), so the
reference points below are approximate bands from common practice, not leaderboard entries.

| reference point | Dice | IoU |
|---|---|---|
| trivial: predict all-background | 0.00 | 0.00 |
| trivial: predict all-foreground | ~0.57 | ~0.42 |
| U-Net from scratch, ~128px (the usual tutorial result) | 0.88 - 0.92 | 0.79 - 0.85 |
| U-Net with an **ImageNet-pretrained encoder** (ResNet-34/EfficientNet), 224-256px | 0.94 - 0.96 | 0.89 - 0.92 |
| large pretrained transformer (SegFormer-B5 / Mask2Former / DINOv2-L + head) | ~0.96+ | ~0.92+ |
| SAM zero-shot with a box prompt (sanity anchor) | - | ~0.90 |

* The pretrained-encoder row is the real bar. Anything from scratch on 3,680 images is
  fighting with one hand tied.
* Pixel accuracy is a weak metric here: pets (pet U boundary) cover ~42% of the pixels, so
  all-background already scores 0.577 pixel accuracy. Read Dice/IoU, not PixAcc.
* Gap read for segmentation differs from classification: train Dice well short of ~0.99 with
  a small train-test gap means **capacity / resolution / prior limited**, not overfitting.

CIFAR_10 (carried over from cifar10.ipynb, kept for the v1.0-v3.0 entries below)
* SOTA training acc = 94-96%
* Good test acc: 85%+
* Gap = around 5%-10% - Not use augmentation.
* Gap = 2%-5% - With augmentation, BatchNorm, dropout, and weight decay.

### Fine-tunning Result

> **Note** — the `v1.0`-`v3.0` entries immediately below are **CIFAR-10** history carried over
> when this notebook was copied from `cifar10.ipynb`. They are kept for reference. The
> Oxford-IIIT Pet lineage starts at **U-Net v1.0** further down.

#### v1.0
Architecture + hyperparameters:

    * Flow = input -> conv(16,3,1,1) -> gelu -> maxpool(2,2) -> conv(32,3,1,1) -> gelu -> dropout(0.2) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> dropout(0.3) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> Flatten shape=(64, 64 * 4 * 4) -> Linear(64 * 4 * 4, 10) -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = LEARNING_RATE

    * # epoch = 20, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: 0.8922, Test Acc: 0.7565

Resnet34

Architecture + hyperparameters:

    * Flow (paper resnet-34 based) = input -> conv(64,7,2,3) -> batchnorm(64) -> relu -> maxpool(3,2,1) -> {residual block 3 x [conv(64,3,1,1) -> batchnorm(64) -> relu -> conv(64,3,1,1) -> batchnorm(64) -> relu] -> residual block 4 x [conv(128,3,2,1) -> batchnorm(128) -> relu -> conv(128,3,1,1) -> batchnorm(128) -> relu] -> residual block 6 x [conv(256,3,2,1) -> batchnorm(256) -> relu -> conv(256,3,1,1) -> batchnorm(256) -> relu] -> residual block 3 x [conv(512,3,2,1) -> batchnorm(512) -> relu -> conv(512,3,1,1) -> batchnorm(512) -> relu]} -> AdaptiveAvgPool2d(1,1) -> Flatten -> [Linear(512, 256) -> gelu -> Linear(256, 128) -> gelu -> Linear(128, 64) -> gelu -> Linear(64, 10)] -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = LEARNING_RATE, weight_decay=WEIGHT_DECAY

    * scheduler => 1e - 6

    * # epoch = 100, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: , Test Acc: 

#### v2.0 — canonical CIFAR ResNet-18 recipe (2026-07-17)

Changes vs the resnet34 v1 run (final: Train Acc 0.9251, Test Acc 0.8247, gap ~10%):

    * Stem: conv(64,3,1,1) + BN + ReLU, NO maxpool — the ImageNet stem (7x7 s2 + maxpool) was
      shrinking 32x32 inputs to 8x8 before the first block; stages now see 32-16-8-4 maps
    * Block: relu2 -> plain ReLU; post-add activation relu(identity + residual); bias-free convs;
      BN (not GroupNorm) in the downsample shortcut
    * Head: single Linear(512, 10) instead of the 512-256-128-64-10 GELU MLP
    * Depth: resnet18 [2,2,2,2] instead of resnet34 [3,4,6,3]
    * Init: Kaiming fan-out + zero-init of each block's last BN gamma (blocks start as identity)
    * Data: RandomCrop(reflect) + HFlip -> ToTensor -> Normalize(CIFAR mean/std) -> RandomErasing(p=0.5);
      removed VerticalFlip, the 2x-1 Lambda, and the in-loop Gaussian input noise
    * Optimiser: SGD lr=0.1, momentum=0.9, nesterov, wd=5e-4 on weights only (none on BN/biases);
      CrossEntropyLoss(label_smoothing=0.1); no gradient clipping
    * Schedule: 5-epoch linear warmup -> cosine to 0; # epoch = 100, batch_size = 128
    * Systems: AMP + channels_last + cudnn.benchmark; seeded (42); per-run TensorBoard dir;
      best/final checkpoints saved to checkpoints/
    * Eval: train accuracy on a clean fixed 10k subset (was: full augmented train set)

Result (verified run, 2026-07-17, RTX 5080, ~3.5 s/epoch, ~6 min total):

    Train Acc: 0.9997, Test Acc: 0.9561 (best), gap 4.4%
    (loss floors near ~0.52 because of label smoothing 0.1 — expected)

#### v3.0 — ViT from scratch, no pretraining (2026-08-12)

Architecture + hyperparameters (3.20M params):

    * Flow = input(3,32,32) -> PatchEmbedding[Conv2d(3,256,k=4,s=4) -> Flatten -> transpose
      -> concat CLS token -> + pos_embed] = (B, 65, 256)
      -> 6 x TransformerEncoderLayer[pre-LN: x + MSA(LN(x)) ; x + MLP(LN(x))]
         MSA = nn.MultiheadAttention(256, 8 heads, head_dim 32, dropout 0.1)
         MLP = Linear(256,512) -> GELU -> Drop(0.1) -> Linear(512,256) -> Drop(0.1)
      -> LayerNorm(256) -> take CLS token -> Linear(256, 10) -> (CrossEntropyLoss)
    * Init: trunc_normal_(std=0.02) on cls_token and pos_embed (learned, not sinusoidal)
    * Optimiser: AdamW, lr=3e-4, weight_decay=5e-2 (applied to ALL params — the decay/no_decay
      split is built in the cell but not passed to the optimizer)
    * Schedule: 5-epoch linear warmup (start_factor 0.1) -> cosine to 0; # epoch = 200, batch_size = 128
    * Loss: CrossEntropyLoss(label_smoothing=0.1) -> loss floors near ~0.50, expected
    * Data: RandomCrop(32, pad 4, reflect) + HFlip -> ToTensor -> Normalize(CIFAR mean/std).
      RandomErasing is COMMENTED OUT for this run (the logged config string still says
      "crop-reflect+hflip+erasing" — stale label, fix before comparing to the ResNet runs)
    * Systems: AMP + channels_last + cudnn.benchmark; seeded (42); per-run TensorBoard dir
    * Eval: train accuracy on the clean fixed 10k train subset

Result (verified run `20260812-174242_ViT_simple`, RTX 5080, ~3.6 s/epoch, ~12 min total):

    Best test acc: 0.8435 (epoch 194) — Train Acc 1.0000, gap ~15.6%
    Final (epoch 200): Loss 0.5057, Train Acc 1.0000, Test Acc 0.8426

Trajectory:

    | epoch | loss   | train  | test   |
    |-------|--------|--------|--------|
    | 50    | 0.7349 | 0.9442 | 0.8160 |
    | 87    | 0.5899 | 0.9909 | 0.8272 |  <- train saturates here
    | 100   | 0.5667 | 0.9918 | 0.8214 |
    | 125   | 0.5345 | 0.9979 | 0.8317 |
    | 150   | 0.5167 | 0.9997 | 0.8362 |
    | 200   | 0.5057 | 1.0000 | 0.8426 |

Read:

    * Underperforms the v2.0 ResNet-18 (95.6% test) by ~11 points, and misses the 85% goal.
      Expected: ViTs have no convolutional locality/translation prior, so on 50k images they
      lean on augmentation and regularization that this run mostly turned off.
    * Train hits 100% by ~epoch 87 while test stalls in the low 82s — the model memorizes the
      train split and the remaining 113 epochs buy only ~1.5 points (cosine decay polish).
      The 15.6% gap is pure overfitting, far outside the 2-5% augmented-regime target.

Next levers (in rough order of expected payoff):

    * Turn RandomErasing back on, and add Mixup/CutMix + RandAugment — the standard
      "ViT on small data" recipe; this is the biggest single gap.
    * Stochastic depth (drop_path ~0.1) across the 6 blocks; raise drop_rate above 0.1.
    * Exclude LayerNorm gains, biases, cls_token and pos_embed from weight decay
      (the decay/no_decay lists are already built — just pass them as param groups).
    * Shrink to depth 4-6 / embed 192 or try patch_size=2 (256 tokens) for finer detail.


### Oxford-IIIT Pet — Fine-tuning Result

#### U-Net v1.0 — U-Net from scratch, no pretraining (2026-08-21)

Architecture + hyperparameters (**31.04M params**, 124.2 MB fp32):

    * Task = binary segmentation, 1 logit per pixel. Oxford trimap {1 pet, 2 background,
      3 boundary} -> target = (trimap != 2), i.e. pet U boundary.  <- see the caveats below
    * Flow = input(3,128,128)
      -> Encoder: 4 x [Block(C_in -> C_out) -> MaxPool(2,2)], C = 64, 128, 256, 512
         Block = conv(3x3, pad 1, bias-free) -> BN -> ReLU -> conv(3x3, pad 1, bias-free) -> BN -> ReLU
      -> Bottleneck: Block(512 -> 1024) at 8x8
      -> Decoder: 4 x [ConvTranspose2d(C, C/2, k=2, s=2) -> concat(skip) -> Block(C -> C/2)]
      -> Conv2d(64, 1, k=1)
    * "same" padding (not the paper's valid convs) + BatchNorm (not in the 2015 paper);
      the centre-crop on the skip is a fallback that never fires at 128px
    * Loss = BCEWithLogitsLoss (plain — no Dice / focal / boundary weighting)
    * Optimiser = AdamW, lr=1e-3, wd=1e-4 on weights only (no decay on 1-D params)
    * Schedule = 5-epoch linear warmup (start_factor 0.1) -> cosine to 0; 100 epochs, batch 32
    * Data = train: Resize(144,144) -> RandomCrop(128) -> HFlip; test: Resize(128,128).
      No colour jitter, rotation, or scale jitter.
    * Systems = AMP + channels_last + cudnn.benchmark; seeded (42)
    * Eval = train Dice on a clean fixed 1,000-image trainval subset; test on the full 3,669
    * `DROP_RATE = 0.0` is declared and logged in the TensorBoard config, but `Unet` takes no
      `drop_rate` argument — it is a dead hyperparameter, not an active setting.

Result (verified run `20260821-170419_Unet_simple`, RTX 5080, ~6.1 s/epoch including both
eval passes, ~10 min total):

    Best test Dice: 0.9367 (epoch 86) — IoU 0.8964, PixAcc 0.9538
    Final (epoch 100): Loss 0.0529, Train Dice 0.9659, Test Dice 0.9360, IoU 0.8950, PixAcc 0.9532
    Dice gap (train - test) = 3.0 points

Trajectory:

    | epoch | loss   | train Dice | test Dice | test IoU | PixAcc |
    |-------|--------|------------|-----------|----------|--------|
    | 1     | 0.4460 | 0.8146     | 0.8146    | 0.7116   | 0.8503 |
    | 10    | 0.2043 | 0.8792     | 0.8746    | 0.8049   | 0.9136 |
    | 20    | 0.1557 | 0.9153     | 0.9063    | 0.8463   | 0.9287 |
    | 30    | 0.1194 | 0.9384     | 0.9241    | 0.8760   | 0.9437 |
    | 40    | 0.1022 | 0.9397     | 0.9153    | 0.8651   | 0.9402 |
    | 50    | 0.0846 | 0.9506     | 0.9315    | 0.8889   | 0.9502 |
    | 60    | 0.0693 | 0.9581     | 0.9343    | 0.8929   | 0.9523 |  <- test plateaus here
    | 70    | 0.0640 | 0.9616     | 0.9351    | 0.8944   | 0.9528 |
    | 80    | 0.0572 | 0.9642     | 0.9358    | 0.8952   | 0.9533 |
    | 86    | 0.0550 | 0.9654     | 0.9367    | 0.8964   | 0.9538 |  <- best
    | 100   | 0.0529 | 0.9659     | 0.9360    | 0.8950   | 0.9532 |

Read:

    * **Clears the from-scratch band (0.88-0.92 Dice) but sits ~1-2 Dice points below the
      pretrained-encoder bar (0.94-0.96)** — and it gets there on an easier-than-standard eval
      protocol (see caveats). So: the U-Net implementation is correct and well-trained; the
      *model* is not yet competitive.
    * The failure mode is **not overfitting**. Train Dice tops out at 0.966 (not ~0.99) with
      only a 3.0-point gap. Compare the v3.0 ViT run — train 1.0000, gap 15.6% — where
      regularization was the lever. Here it is capacity / resolution / pretraining. Adding
      dropout or heavier augmentation to *this* run would make it worse.
    * **40% of the run was wasted.** Test Dice is 0.9343 at epoch 60 and 0.9360 at epoch 100:
      the last 40 epochs bought 0.2 points. Cut to ~60 epochs and spend the saved compute on
      resolution instead.
    * Loss and metric disagree about progress: BCE falls smoothly to 0.053 while test Dice is
      flat from epoch 60. BCE optimises per-pixel likelihood on the easy interior, not region
      overlap at the boundary — which is where all the remaining error lives.

Caveats — **revised 2026-08-24 after measuring them** (`oxford_pet_analysis.ipynb` S5).
Three of the four originally listed here were wrong; the corrections are below.

    1. **The boundary class is folded into the positive target.** Trimap class 3 is
       "not classified" — the annotators declined to label it. `(mask != 2)` makes it pet.
       Measured over the 3,669 test masks, the band is **12.2% of all pixels at native
       resolution** and **29.0% of the positive target**.
       ~~This inflates Dice by 2-4 points.~~ **WRONG, and backwards.** Re-scoring the same
       weights: 0.9367 with the band as pet, **0.9418** with it ignored. Excluding it *raises*
       the score by 0.5 points, because the band is the hard region — dropping it removes the
       pixels the model is worst at.
       The real caveat is bigger and different: scored against the animal's **actual outline**
       (`boundary="background"`), this model gets **0.7801**, a 15.6-point drop. It predicts
       pet-plus-band because that is what it was trained on. Report 0.9418 against papers that
       ignore the void band; never claim 0.9367 means the model delineates pets.
    2. ~~**Evaluated at 128x128, not native resolution** — all remaining error is boundary
       error, heavily quantised at 128px.~~ **WRONG.** Upsampling the logits to native
       resolution before thresholding changes Dice by **under 0.001** (0.9362 -> 0.9371 on a
       400-image subset). Immaterial. Raising `image_size` is still a model lever; it is not a
       measurement-honesty issue.
    3. **IoU is dataset-aggregate (micro), not mean-per-image.** Still true. Micro IoU is
       dominated by large pets; most papers report the per-image mean. Dice here *is* per-image
       averaged, so the two logged numbers use different aggregation conventions. The pipeline
       now reports both (`iou` and `mean_iou`).
    4. **Train/test scale mismatch.** Still true. Train does `Resize(144,144) -> RandomCrop(128)`
       (aspect ratio destroyed, then an effective 1.125x zoom); test does `Resize(128,128)`
       (no zoom). The two splits see different scale statistics.

Also measured, and not in the original list:

    5. **The model is well calibrated for the protocol it was trained on.** Under
       `boundary="foreground"`, threshold 0.5 is exactly optimal — there is no free Dice on the
       table. Under `boundary="ignore"` the optimum runs up to 0.95 (0.9628 Dice), but that is
       the threshold absorbing a train/test protocol *mismatch* — peeling off a band the model
       was explicitly taught to predict — not a generalization gain. It is also swept on the
       test split, so it is not a reportable number. The principled fix is to *train* with
       `boundary="ignore"`. hflip TTA is a genuine free +0.33.
    6. **The error is not concentrated at the boundary.** Error *rate* within 5-7px of the
       outline is ~7x the far-field rate, but the far field is 44% of pixels, so most error
       *mass* is far from any edge. The signature is 1% of images failing wholesale (per-image
       Dice p1 = 0.50, median 0.97), not uniformly fuzzy edges. This demotes boundary-weighted
       losses and promotes the pretrained encoder.

Next levers (rough order of expected payoff):

    1. **ImageNet-pretrained encoder** — ResNet-34/50, EfficientNet, or ConvNeXt in place of the
       from-scratch encoder, keeping the U-Net decoder. Single biggest lever on 3,680 images:
       expect +2-4 Dice and much faster convergence. This is what the 0.94-0.96 band assumes.
    2. **Train and evaluate at 224 or 256px.** ~3-4x compute (still only ~25 s/epoch on the
       5080) for +1-2 Dice, and it makes the numbers comparable to the literature.
    3. **Retrain with `boundary="ignore"`** rather than threshold-patching these weights
       (caveat 5) — already the `oxford_pet` preset default. BCE + soft Dice is still worth
       having (also the preset default), but demoted: caveat 6 shows the boundary carries only
       ~13% of the error mass, so a region/boundary term buys less than assumed.
    4. **Fix the protocol** (correctness, not score — this will *lower* the number, and should):
       exclude trimap==3 from both loss and metric via `ignore_index`, or move to the 3-class
       trimap task; upsample logits to native resolution before scoring; report mean-per-image
       IoU alongside micro IoU.
    5. **Real augmentation**: `RandomResizedCrop(128, scale=(0.7, 1.0))` in place of
       Resize+RandomCrop (fixes caveat 4 and adds scale variety), plus +/-15 deg rotation,
       ColorJitter, and light affine.
    6. **Cheap architectural wins**: bilinear upsample + 3x3 conv instead of ConvTranspose2d
       (removes checkerboard artifacts); deep supervision on the decoder stages; a small ASPP
       or attention gate at the bottleneck for global context.
    7. **Free at eval**: weight EMA, and hflip test-time augmentation (~+0.2-0.5 Dice).
    8. **Not levers**: more epochs, more dropout, more weight decay. The gap is 3 points.

Next dataset: **Pascal VOC 2012 segmentation** (21 classes, 1,464 train / 1,449 val, or 10,582
with the SBD augmented set; metric mIoU). It is the canonical next rung after a binary task and
forces the machinery worth having — multi-class CrossEntropy with `ignore_index=255` for the
void border, severe class imbalance, and per-class mIoU. It also settles the pretraining
question directly: from-scratch VOC lands in the ~50s mIoU, pretrained-backbone DeepLab/U-Net
in the mid-70s and above. Fits an RTX 5080 comfortably.
Then **Cityscapes** (19 classes, 2,975 train at 1024x2048 — high resolution, real-world scenes;
needs registration to download), and **ADE20K** (150 classes, 20k train) after that.

In [21]:
data = [1,3,3,3,2,2,2,4]
temp = sorted(data)
max(temp, key=data.count)

2